In [1]:
'''# How to run this code in Google Colab or Local Machine: 
#Before running this code, make sure to install the required python and needed packages. 
# Since Iam used to Visual studio code, below instructions are for VSC but can be true fot other similar IDEs.
# If you don't want to install them in Local Machine then you use the colab directly. 
# Choose Runtime -> Python 3(ipykernel)
# If the above is not available, Select another kernel -> Colab -> Auto Conect or New Colab server if you do not have

'''

import sys
from datetime import datetime
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Runtime is running Date and time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))



Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Executable: /usr/bin/python3
Runtime is running Date and time: 2026-10-31 12:12:18


In [2]:
''' Here we are using the Gdrive to store the file and read it from there. 
If you want to use the local machine or colab folder, you can use the commented code below to read the file.
'''
# If the file is stored in the local machine or in Colab folder, you can use the following code to read the file.
# file_path = r"C:\Users\veera\AI\SampleFile\test-product-review.parquet"
# file_path = r"\sample_data\test-product-review.parquet"

from google.colab import drive
import os
drive.mount('/content/drive')
print("Drive mounted:", os.path.ismount('/content/drive'))

file_path = '/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet'
print("File exists:"+file_path+" -", os.path.isfile(file_path))




Mounted at /content/drive
Drive mounted: True
File exists:/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet - True


In [3]:
# Read the Parquet file 
# with specific columns and filter for English language reviews
# For better understanding I have filtered the data for few columns English language reviews only. You can remove the filter if you want to read all the data.

from time import time

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
                      filters=[[('language', '==', 'en')]])

# #Print the first few rows of the DataFrame to see how the data looks like.
df = table.to_pandas()
# df.info()
# print(df.head(4))
#print(df.shape)
print("Count of each star rating:"+str(df["stars"].value_counts().to_dict()))

Count of each star rating:{1: 104, 5: 92, 3: 90, 4: 90, 2: 84}


In [4]:
'''DATA PRE-PROCESSING
"tokens", "lemmas", "stems"
For cleaning the data we will use all the 3 above techniques. 
While doing the execution fount that NOT and NEGATION words are important for sentiment analysis. So we will keep them in the stopwords list.
Along with that we will also remove the URLs, special characters, and extra spaces.
After applying the above techniques we will create a new column called "review_body_processed" by combining all the words(tokens) which will be used for vectorization and model training.
'''
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download these resources once
nltk.download("punkt")
nltk.download("punkt_tab")  # Needed by some NLTK versions
nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
# Keep negation words because they can change review sentiment
stop_words -= { "no", "not", "never", "none", "neither", "nor", "noone", "nobody", "nothing", "nowhere", "hardly", "scarcely", "barely", "seldom", "rarely", "don't", "doesn't", "didn't", "isn't", "aren't", "wasn't", "weren't", "haven't", "hasn't", "hadn't", "won't", "wouldn't", "shouldn't", "can't", "cannot", "couldn't", }

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def preprocess_review(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]

    # Apply lemmatization, then stemming
    lemmas = [
        lemmatizer.lemmatize(
            lemmatizer.lemmatize(token, pos="v"),
            pos="n"
        )
        for token in tokens
    ]
    stems = [stemmer.stem(token) for token in lemmas]

    return tokens, lemmas, stems

df[["tokens", "lemmas", "stems"]] = df["review_body"].apply(
    lambda text: __import__("pandas").Series(preprocess_review(text))
)

# Text column to use later for vectorization/model training
df["review_body_processed"] = df["stems"].apply(" ".join)

# print(df[["review_body", "tokens", "lemmas", "stems"]].head())
print(df[["review_body", "review_body_processed"]].head())
df.info()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


                                         review_body  \
0  Well I haven't stunned anyone...yet. So I can ...   
1  I loved the first book so much had to read the...   
2  This wasn't my favorite bag, but it wasn't my ...   
3  Good storyline but the characterizations seeme...   
4  Never noticed a big difference than a normal b...   

                               review_body_processed  
0  well n't stun anyon yet rate know buy small co...  
1  love first book much read sequel actual could ...  
2  n't favorit bag n't least favorit either bag l...  
3  good storylin character seem littl lack 'm def...  
4                never notic big differ normal brush  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460 entries, 0 to 459
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   review_id              460 non-null    object
 1   review_body            460 non-null    object
 2   language            

In [5]:
# Install if needed:
# pip install pandas pyarrow scikit-learn

import re
import pandas as pd
import pyarrow.parquet as pq


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC



In [6]:
'''DATA CLEANING AND LABEL PREPARATION
Here we will convert the "stars" column to numeric and create a new "sentiment" column based on the star ratings.
We wil keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
We might also need to drop and empty or null values in the review_body"stars" column, as they cannot be used for sentiment analysis. Since Data is clean commenting it out. If you want to use it, you can uncomment the code below.
'''

# df = df[["review_body", "stars"]].dropna().copy()

# Convert stars to numeric as had problems with some non-numeric values in the dataset. Coerce errors to NaN, then drop those rows.
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
# df = df.dropna(subset=["stars"]).reset_index(drop=True)

# Keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["stars"].apply(create_sentiment_label)
df_deep = df.copy()  # Create a copy of the DataFrame for further processing
# # Remove neutral examples for a simpler binary sentiment task
# df = df[df["sentiment"] != "neutral"].reset_index(drop=True)

print(df[["stars", "sentiment"]].head(6))
print(df["stars"].value_counts())
print(df["sentiment"].value_counts())


   stars sentiment
0      2  negative
1      1  negative
2      3   neutral
3      3   neutral
4      3   neutral
5      5  positive
stars
1    104
5     92
3     90
4     90
2     84
Name: count, dtype: int64
sentiment
negative    188
positive    182
neutral      90
Name: count, dtype: int64


In [7]:
# 2. Train/Test Split (stratified across the 3 sentiment classes)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["review_body_processed"],
    df["sentiment"],
    test_size=0.3,
    random_state=42,
   stratify =df["sentiment"],
)

# 3. Feature Extraction (TF-IDF with unigrams and bigrams)
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print ("Train shape (Rows + columns of words):", X_train.shape)
print ("Test shape (Rows + columns of words):", X_test.shape)
print("Class distribution in train:", y_train.value_counts().to_dict())

Train shape (Rows + columns of words): (322, 6464)
Test shape (Rows + columns of words): (138, 6464)
Class distribution in train: {'negative': 132, 'positive': 127, 'neutral': 63}


In [8]:
''' Logistic Regression Model Training and Evaluation
We will train a Logistic Regression model on the training data and evaluate its performance on the test data
'''
from sklearn.metrics import accuracy_score, classification_report

# Initialize and Train Logistic Regression
# Make sure class_weight is set to "balanced" to handle neutral class imbalance. This will help the model to give equal importance to all classes during training. If not this will lead to bias towards the majority class and poor performance on the neutral sentiment.
log_reg = LogisticRegression(class_weight="balanced")
log_reg.fit(X_train, y_train)

# Predict and Evaluate
y_pred = log_reg.predict(X_test)
print("--- Logistic Regression Results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

# Before doing this we will create array to store the results of the models for comparison later. We will create a list of dictionaries to store the model name, accuracy, precision, recall, and F1-score for each model.
results = []
model_names = []
def get_metrics(y_true, y_pred):
  acc = accuracy_score(y_true, y_pred)
  prec, rec, f1, _ = precision_recall_fscore_support(
      y_true, y_pred, average="weighted", zero_division=0
  )
  return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1}

results.append(get_metrics(y_test, y_pred))
model_names.append("LogisticRegression ")


--- Logistic Regression Results ---
Accuracy: 0.6449
              precision    recall  f1-score   support

    negative       0.69      0.71      0.70        56
     neutral       0.29      0.15      0.20        27
    positive       0.68      0.82      0.74        55

    accuracy                           0.64       138
   macro avg       0.55      0.56      0.55       138
weighted avg       0.61      0.64      0.62       138



In [9]:
# Checking the C  value of the model to see how well it is performing. The C value is a regularization parameter that controls the trade-off between achieving a low error on the training data and minimizing the model complexity. 
# A smaller C value means stronger regularization, which can help prevent overfitting but may lead to underfitting if set too high. 
# A larger C value means weaker regularization, which can lead to overfitting if set too low. 
# The optimal C value depends on the specific dataset and problem being solved, and it is often determined through cross-validation or hyperparameter tuning.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

# 1. Define the parameter grid to search over
param_grid = {
    "C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "class_weight": [None, "balanced"],
}

# 2. Initialize the base Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)

# 3. Set up GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring="f1_weighted",  # Optimize for weighted F1-score to handle 3 classes
    n_jobs=-1,  # Use all available CPU cores
)

# 4. Fit the grid search on the training data
grid_search.fit(X_train, y_train)

# 5. Print the best parameters found
print("Best Parameters Found:")
print(grid_search.best_params_)
print(f"Best Cross-Validation F1-Score: {grid_search.best_score_:.4f}\n")

# 6. Evaluate the best model on the unseen test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("--- Evaluation of Tuned Model (Logistic Regression - Hyperparameter Tuning)---")
print(classification_report(y_test, y_pred, zero_division=0))

Best Parameters Found:
{'C': 1.0, 'class_weight': None}
Best Cross-Validation F1-Score: 0.5567

--- Evaluation of Tuned Model (Logistic Regression - Hyperparameter Tuning)---
              precision    recall  f1-score   support

    negative       0.64      0.80      0.71        56
     neutral       0.00      0.00      0.00        27
    positive       0.68      0.84      0.75        55

    accuracy                           0.66       138
   macro avg       0.44      0.55      0.49       138
weighted avg       0.53      0.66      0.59       138



In [10]:
'''Lets do 2 more model fit and evaluation in CLassification task. We will use Multinomial Naive Bayes and Linear SVC models for this purpose.
'''


results.append(get_metrics(y_test, y_pred))
model_names.append("LogisticRegression Hyperparameter Tuned")

# "Linear SVM (LinearSVC)" Fit , predict and evaluate
Linear_SVC = LinearSVC(random_state=42)
Linear_SVC.fit(X_train, y_train)
y_pred = Linear_SVC.predict(X_test)
results.append(get_metrics(y_test, y_pred))
model_names.append("LinearSVC")

# Multinomial Naive Bayes
MNB=MultinomialNB()
MNB.fit(X_train, y_train)
y_pred = MNB.predict(X_test)
results.append(get_metrics(y_test, y_pred))
model_names.append("MultinomialNB")

comparison_df = pd.DataFrame(results, index=model_names)

print("=" * 65)
print("             SIDE-BY-SIDE MODEL PERFORMANCE COMPARISON           ")
print("=" * 65)
print(comparison_df.round(4))
print("=" * 65)


             SIDE-BY-SIDE MODEL PERFORMANCE COMPARISON           
                                         Accuracy  Precision  Recall  F1-Score
LogisticRegression                         0.6449     0.6075  0.6449    0.6194
LogisticRegression Hyperparameter Tuned    0.6594     0.5305  0.6594    0.5880
LinearSVC                                  0.6449     0.5962  0.6449    0.5991
MultinomialNB                              0.6087     0.4899  0.6087    0.5428


In [17]:
'''Deep Learning Model Training and Evaluation 
we will use the Hugging Face Transformers library to fine-tune a pre-trained BERT model for sentiment classification.
'''

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import logging
from transformers import logging as transformers_logging

# Suppress Hugging Face info/warning logs and load reports
transformers_logging.set_verbosity_error()
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

from transformers import (
    DataCollatorWithPadding,
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments,
)
!pip install -q optuna
import optuna  # Required for hyperparameter search

# 1. Dataset Setup (Assumes 'df' with 'review_body' and 'sentiment' columns is defined)
# df_deep already defined in the previous code cell. We will use it for deep learning model training and evaluation.

# table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
#                       filters=[[('language', '==', 'en')]])
table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
                      )
# #Print the first few rows of the DataFrame to see how the data looks like.
df_deep = table.to_pandas()

print("Count of each star rating:"+str(df_deep["stars"].value_counts().to_dict()))

# Convert stars to numeric as had problems with some non-numeric values in the dataset. Coerce errors to NaN, then drop those rows.
df_deep["stars"] = pd.to_numeric(df_deep["stars"], errors="coerce")
# df = df.dropna(subset=["stars"]).reset_index(drop=True)

# Keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df_deep["sentiment"] = df_deep["stars"].apply(create_sentiment_label)


# 2. Map string labels to integers (Required for Transformer models)
label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
inv_label_mapping = {0: "negative", 1: "neutral", 2: "positive"}
df_deep["label"] = df_deep["sentiment"].map(label_mapping)

# 3. Train/Test Split (Stratified)
train_df, test_df = train_test_split(
    df_deep, test_size=0.3, random_state=42, stratify=df_deep["label"]
)

# Convert Pandas DataFrames to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))

Count of each star rating:{5: 323, 1: 302, 2: 294, 3: 292, 4: 289}
Train dataset size: 1050
Test dataset size: 450


In [18]:
# 4. Tokenization using Multilingual DistilBERT Tokenizer
model_name = "distilbert-base-multilingual-cased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)


def tokenize_function(examples):
  return tokenizer(examples["review_body"], truncation=True, max_length=128)


tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print(f"Number of training samples: {len(tokenized_train)}")
print(f"Number of test samples: {len(tokenized_test)}")
print("Sample tokenized features:", tokenized_train[0]["input_ids"])


# 5. Model Initialization Function (Required for Optuna hyperparameter search)
def model_init(trial):
  return DistilBertForSequenceClassification.from_pretrained(
      model_name, num_labels=3, id2label=inv_label_mapping, label2id=label_mapping
  )


# 6. Define Evaluation Metric Function
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  preds = np.argmax(predictions, axis=1)
  accuracy = np.mean(preds == labels)
  return {"accuracy": accuracy}


# 7. Configure Training Arguments
training_args = TrainingArguments(
    output_dir="./distilbert_multilingual_results",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    fp16=True,
    dataloader_num_workers=2,
    per_device_eval_batch_size=32,
)

# 8. Initialize Trainer with model_init
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)


# 9. Define Hyperparameter Search Space for Optuna
def hp_space(trial):
  return {
      "learning_rate": trial.suggest_float(
          "learning_rate", 1e-5, 5e-5, log=True
      ),
      "per_device_train_batch_size": trial.suggest_categorical(
          "per_device_train_batch_size", [4, 8, 16]
      ),
      "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
      "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
  }


# 10. Run Hyperparameter Tuning
print("Starting Optuna Hyperparameter Search...")
best_run = trainer.hyperparameter_search(
    direction="maximize", backend="optuna", hp_space=hp_space, n_trials=5
)

print("\nBest Hyperparameters Found:")
print(best_run.hyperparameters)

# 11. Apply Best Hyperparameters & Finalize Training
print("\nApplying Best Hyperparameters for Final Training...")
for key, value in best_run.hyperparameters.items():
  setattr(trainer.args, key, value)

trainer.train()

# 12. Evaluate on the Test Set
from sklearn.metrics import precision_recall_fscore_support

print("\nEvaluating Best Model on Test Data:")
eval_results = trainer.evaluate(tokenized_test)
print(eval_results)

# Extract predictions for Multilingual DistilBERT to compute metrics
preds_output = trainer.predict(tokenized_test)
preds = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels, preds, average="weighted"
)
accuracy = eval_results.get("eval_accuracy", 0.0)

# Append results using a distinct name to reflect the multilingual model
results.append(
    {"Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1}
)
model_names.append("DistilBERT-Multilingual")

# Create and display the unified comparison DataFrame
comparison_df = pd.DataFrame(results, index=model_names)

print("=" * 65)
print("            SIDE-BY-SIDE MODEL PERFORMANCE COMPARISON           ")
print("=" * 65)
print(comparison_df.round(4))
print("=" * 65)

Map:   0%|          | 0/1050 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Number of training samples: 1050
Number of test samples: 450
Sample tokenized features: [101, 10141, 70324, 10554, 28552, 10112, 10801, 10326, 10109, 53393, 68228, 28311, 12904, 10168, 15537, 15699, 26232, 117, 66355, 48497, 15699, 10563, 14866, 10121, 10141, 70324, 11364, 119, 113, 18919, 38171, 25680, 10112, 114, 178, 112, 11346, 23271, 10152, 124, 10110, 10196, 35869, 18348, 10110, 13508, 10119, 10460, 10119, 28552, 12276, 10326, 16920, 10109, 16004, 117, 10614, 37575, 119, 14375, 10119, 33478, 10526, 10106, 19065, 10284, 10355, 10911, 81010, 10112, 10231, 13224, 10104, 10563, 10322, 30366, 11989, 119, 169, 11915, 23907, 66355, 11504, 35407, 18289, 107, 36823, 27986, 107, 78323, 17364, 11684, 10121, 17331, 169, 34372, 57822, 119, 91985, 97184, 34819, 102]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-10-31 12:32:00,526] A new study created in memory with name: no-name-2725b79f-a1a3-4e89-bc54-23a125f5de5e


...

{'eval_loss': '0.8773', 'eval_accuracy': '0.6467', 'eval_runtime': '0.3346', 'eval_samples_per_second': '1345', 'eval_steps_per_second': '44.83', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.7129', 'grad_norm': '6.452', 'learning_rate': '1.185e-06', 'epoch': '3.788'}
{'eval_loss': '0.893', 'eval_accuracy': '0.6311', 'eval_runtime': '0.331', 'eval_samples_per_second': '1360', 'eval_steps_per_second': '45.32', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '40.43', 'train_samples_per_second': '103.9', 'train_steps_per_second': '13.06', 'train_loss': '0.6974', 'epoch': '4'}

Evaluating Best Model on Test Data:
{'eval_loss': '0.8624', 'eval_accuracy': '0.6422', 'eval_runtime': '0.326', 'eval_samples_per_second': '1381', 'eval_steps_per_second': '46.02', 'epoch': '4'}
{'eval_loss': 0.8623504042625427, 'eval_accuracy': 0.6422222222222222, 'eval_runtime': 0.326, 'eval_samples_per_second': 1380.515, 'eval_steps_per_second': 46.017, 'epoch': 4.0}
            SIDE-BY-SIDE MODEL PERFORMANCE COMPARISON           
                                         Accuracy  Precision  Recall  F1-Score
LogisticRegression                         0.6449     0.6075  0.6449    0.6194
LogisticRegression Hyperparameter Tuned    0.6594     0.5305  0.6594    0.5880
LinearSVC                                  0.6449     0.5962  0.6449    0.5991
MultinomialNB                              0.6087     0.4899  0.6087    0.5428
DistilBERT                   

In [19]:
# 1. Create a evaluation copy of your test DataFrame
eval_df = test_df.copy()

# 2. Assign the model predictions back to the DataFrame
eval_df["predicted_label"] = preds

# 3. Compute accuracy and sample size for each language group
language_performance = []

for lang, group in eval_df.groupby("language"):
  acc = accuracy_score(group["label"], group["predicted_label"])
  sample_count = len(group)
  language_performance.append(
      {"Language": lang, "Accuracy": acc, "Samples": sample_count}
  )

# 4. Convert to DataFrame and sort by the most frequent languages first
lang_perf_df = pd.DataFrame(language_performance).sort_values(
    by="Samples", ascending=False
)

# 5. Display the performance breakdown table
print("=" * 55)
print("           MODEL PERFORMANCE BREAKDOWN BY LANGUAGE          ")
print("=" * 55)
print(lang_perf_df.to_string(index=False))
print("=" * 55)

           MODEL PERFORMANCE BREAKDOWN BY LANGUAGE          
Language  Accuracy  Samples
      fr  0.679487      156
      de  0.644737      152
      en  0.598592      142
